# HAGI - HF Setup (Colab): publish data + refresh model cards

Run this **once, on Colab**, before Kaggle. Colab mounts Google Drive natively, so it's
the natural bridge: it reads your tokenized shards from Drive and pushes them to an **HF
dataset repo** (with a data card), then refreshes the four ablation **model cards**.
After this, every environment (Kaggle, local) pulls the data straight from HF - no Drive
access, no re-tokenizing.

## Before running
1. **Secrets (key icon, left sidebar) -> add `HF_TOKEN`** with **write** access, toggle
   *Notebook access* on. Never printed or committed.
2. Your tokenized `shard_*.bin` files are somewhere under your mounted Drive.
3. Runtime can be **CPU** - this only moves files and text.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone repo + install the Hub client

In [ ]:
import os
%cd /content
if not os.path.isdir('HAGI'):
    !git clone -b experimental https://github.com/ShmidtS/HAGI.git
%cd /content/HAGI
!git pull --ff-only origin experimental
!pip install -q -U huggingface_hub
print('cwd:', os.getcwd())

## 3. Knobs + login
Set `DRIVE_DATA` to the folder that holds your `shard_*.bin` files. Confirm the shards
are the **SmolLM2** tokenizer (vocab 49,152) - anything else is incompatible.

In [ ]:
# ---- knobs ----
USER         = 'NAME0x0'
DRIVE_DATA   = '/content/drive/MyDrive/hagi-data'      # folder with shard_*.bin
DATASET_NAME = 'hagi-fineweb-edu-smollm2'              # HF dataset repo name under USER
PUBLIC       = False                                  # False = private (recommended); flip later in HF UI
SKIP_DATA    = False                                  # True = skip Drive/dataset cells, only refresh cards
# Geo verdict is BAKED INTO push_model_cards.py (GEO_DIAG: seeds=[1], D-B=+0.0210,
# Dnogeo-B=+0.0176 -> grade machinery itself hurts). Leave None to use it; set only to override.
GEO          = None
# ---------------
import os, glob
from huggingface_hub import login, whoami
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))
print('logged in as:', whoami()['name'])
if not SKIP_DATA:
    n = len(glob.glob(os.path.join(DRIVE_DATA, '*.bin')))
    assert n > 0, f'no *.bin in {DRIVE_DATA} - fix DRIVE_DATA (or set SKIP_DATA=True)'
    print(f'found {n} shard(s) in {DRIVE_DATA}')
else:
    print('SKIP_DATA=True -> skip cells 4-5, go straight to cell 6 (cards only)')

## 4. Preview the data card (no upload)

In [ ]:
!python scripts/push_dataset.py --user {USER} --data "{DRIVE_DATA}" --repo-name {DATASET_NAME} --dry-run | head -55

## 5. Push the dataset (card + shards)
First run uploads all shards over LFS - can take a while for multi-GB data. Re-running
only sends changed files.

In [ ]:
vis = '--public' if PUBLIC else ''
!python scripts/push_dataset.py --user {USER} --data "{DRIVE_DATA}" --repo-name {DATASET_NAME} {vis}

## 6. Refresh the four model cards (cross-linked + geometry verdict)
Each card now carries the **Geometry-diagnostic** section by default (B vs D vs
D_nogeo on held-out, with the honest read). Leaves `hagi-stage0` untouched (links
to it only). If the dataset is already on HF, set `SKIP_DATA=True` in cell 3 and
run only cells 1-3 and this one.

In [ ]:
import json
geo_arg = []
if GEO:
    json.dump(GEO, open('/content/geo.json', 'w'))
    geo_arg = ['--geo-json', '/content/geo.json']
    print('cards will include the OVERRIDE geo verdict:', GEO)
else:
    print('using the baked-in GEO_DIAG verdict (D-B=+0.0210, Dnogeo-B=+0.0176)')
import subprocess
subprocess.run(['python', 'scripts/push_model_cards.py', '--user', USER, *geo_arg], check=True)
print('\nlinks:')
print(f'  data:    https://huggingface.co/datasets/{USER}/{DATASET_NAME}')
for m in ['a', 'b', 'c', 'd']:
    print(f'  model {m.upper()}: https://huggingface.co/{USER}/hagi-ablation-{m}')